# Parse Dataset — chotot

Pipeline làm sạch dữ liệu bất động sản từ Chợ Tốt.  
Input: `../data/chotot_raw.csv`  
Output: `../data/chotot_final.csv` (file duy nhất)

**Flow:**
1. Setup & load data
2. Parse numeric columns (price, area, dimensions, counts)
3. Parse location
4. Clean categorical columns
5. Post-processing & export


In [1]:
pip install ipykernel pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup

In [2]:
# !pip install ipykernel pandas numpy matplotlib scikit-learn
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

INPUT_PATH   = Path("../data/chotot_raw.csv")
OUTPUT_FINAL = Path("../data/chotot_final.csv")

df = pd.read_csv(INPUT_PATH)
print("Shape:", df.shape)
df.head(3)

Shape: (9004, 29)


,title,price,area,location,description,Diện tích đất:,Giá/m2:,Hướng cửa chính:,Giấy tờ pháp lý:,Đặc điểm nhà/đất:,Loại hình đất:,Chiều ngang:,Chiều dài:,Số phòng ngủ:,Số phòng vệ sinh:,Loại hình nhà ở:,Tình trạng nội thất:,Diện tích sử dụng:,Tình trạng bất động sản:,Diện tích:,Loại hình căn hộ:,Tổng số tầng:,Tên phân khu/Lô/Block/Tháp:,Mã căn / Mã căn hộ:,Tầng số:,Hướng ban công:,Mã lô:,Đặc điểm căn hộ:,Loại hình văn phòng:
0,Bán lô đất đường Lỗ Giáng 8 - Hoà Xuân giá đầu,"2,38 tỷ- 100 m2",- 100 m2,"Đường Lỗ Giáng 8, Phường Hòa Xuân, Quận Cẩm Lệ, Đà Nẵng",Còn lô giá rẻ nhất khu vực: Nam Cẩm Lệ\n✔️ Đường Lỗ Giáng 8 - Hoà Xuân \n✔Vị trí song song với đường Mẹ Thứ \n✔Diện ...,100 m2,"23,8 triệu/m2",Nam,Đã có sổ,Mặt tiền,Đất thổ cư,5 m,20 m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà mặt tiền Cao Đạt 1 trệt 2 lầu trung tâm quận 5,18 tỷ- 79 m2,- 79 m2,"Đường Cao Đạt, Phường 1, Quận 5, Tp Hồ Chí Minh","Nhà 1 trệt 2 lầu\nDiện tích 4,15x18,8\n4 phòng ngủ 2 toilet 1 khách\nNội thất cơ bản đầy đủ\nMặt tiền trước nhà rộng...",79 m²,"227,85 triệu/m²",Nam,Đang chờ sổ,Mặt tiền,Đất thổ cư,4 m,18 m,4 phòng,3 phòng,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,89 m²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Đất ngay tthc bầu bàng, giá chỉ 2tr 1m2",1 tỷ- 500 m2,- 500 m2,"Đường Quốc Lộ 13, Thị trấn Lai Uyên, Huyện Bàu Bàng, Bình Dương","Vài lô liền kề nằm ngay kcn , tthc bầu bàng \nDiện tích rộng nên có thể đầu tư xây trọ cách các khu công nghiệp chỉ...",500 m2,2 triệu/m2,Nam,Đã có sổ,Mặt tiền,Đất nông nghiệp,5 m,100 m,4 phòng,3 phòng,"Nhà mặt phố, mặt tiền",Nội thất đầy đủ,89 m²,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Parse Numeric Columns

Các cột numeric cần parse:
`price`, `area`, `Diện tích đất:`, `Giá/m2:`, `Chiều ngang:`, `Chiều dài:`,
`Số phòng ngủ:`, `Số phòng vệ sinh:`, `Diện tích sử dụng:`, `Diện tích:`,
`Tổng số tầng:`, `Mã căn / Mã căn hộ:`, `Tầng số:`, `Mã lô:`


In [3]:
# Helper functions

def normalize_text(value):
    """Chuẩn hóa text cơ bản trước khi parse số."""
    if pd.isna(value):
        return None
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("m²", "m2").replace("㎡", "m2")
    text = text.replace("tỉ", "ty").replace("tỷ", "ty")
    text = re.sub(r"\btriệu\b|\btrieu\b", "trieu", text)
    text = re.sub(r"(?<=\d)\s*tr\b", "trieu", text)  # "3tr" → "3trieu"
    text = text.replace(",", ".")
    return text


def first_number(text):
    """Lấy số đầu tiên trong chuỗi đã normalize."""
    if text is None:
        return np.nan
    match = re.search(r"\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else np.nan


def parse_price_vnd(value):
    """Parse giá về VNĐ (int)."""
    text = normalize_text(value)
    if not text:
        return np.nan
    if any(k in text for k in ["thoa thuan", "thỏa thuận", "thoả thuận", "lien he", "liên hệ"]):
        return np.nan

    price_part = text.split("-")[0].strip()

    # Dạng "X tỷ Y triệu"
    m = re.search(r"(\d+(?:\.\d+)?)\s*ty\s+(\d+(?:\.\d+)?)", price_part)
    if m:
        return int(round(float(m.group(1)) * 1_000_000_000 + float(m.group(2)) * 1_000_000))

    number = first_number(price_part)
    if pd.isna(number):
        return np.nan
    if "ty" in price_part:
        return int(round(number * 1_000_000_000))
    if "trieu" in price_part:
        return int(round(number * 1_000_000))
    if "đ" in price_part:
        raw = re.search(r"[\d\.]+", price_part)
        if raw:
            return int(raw.group(0).replace(".", ""))
    return np.nan


def parse_area_m2(value):
    """Parse diện tích về m² (float). Hỗ trợ ha, công, sào, mẫu."""
    text = normalize_text(value)
    if not text:
        return np.nan
    text = text.replace("-", " ").strip()

    for pattern, factor in [
        (r"(\d+(?:\.\d+)?)\s*ha\b",   10_000),
        (r"(\d+(?:\.\d+)?)\s*công\b",  1_000),
        (r"(\d+(?:\.\d+)?)\s*sào\b",     360),
        (r"(\d+(?:\.\d+)?)\s*mẫu\b",   3_600),
    ]:
        m = re.search(pattern, text)
        if m:
            return float(m.group(1)) * factor

    text = text.replace("m2", " ").replace("m²", " ").replace("m", " ")
    m = re.search(r"\d+(?:[\.,]\d+)?", text)
    if not m:
        return np.nan
    num = m.group(0)
    if re.match(r"^\d+\.\d{3}$", num):
        num = num.replace(".", "")
    else:
        num = num.replace(",", ".")
    return float(num)


def parse_price_per_m2(value):
    """Parse đơn giá về VNĐ/m² (int)."""
    text = normalize_text(value)
    if not text:
        return np.nan
    if any(k in text for k in ["thoa thuan", "thỏa thuận", "thoả thuận", "lien he", "liên hệ"]):
        return np.nan

    text = re.sub(r"/\s*m2", "", text).strip()
    price_part = text.split("-")[0].strip()

    m = re.search(r"(\d+(?:\.\d+)?)\s*ty\s+(\d+(?:\.\d+)?)", price_part)
    if m:
        return int(round(float(m.group(1)) * 1_000_000_000 + float(m.group(2)) * 1_000_000))

    number = first_number(price_part)
    if pd.isna(number):
        return np.nan
    if "ty" in price_part:
        return int(round(number * 1_000_000_000))
    if "trieu" in price_part:
        return int(round(number * 1_000_000))
    if "đ" in price_part or "vnd" in price_part:
        raw = re.search(r"[\d\.]+", price_part)
        if raw:
            return int(raw.group(0).replace(".", ""))
    return np.nan


def parse_meter(value):
    """Parse chiều ngang/chiều dài về mét (float)."""
    text = normalize_text(value)
    return first_number(text) if text else np.nan


def parse_count(value):
    """Parse số phòng/tầng về int dương."""
    text = normalize_text(value)
    if not text:
        return np.nan
    number = first_number(text)
    if pd.isna(number) or number < 0:
        return np.nan
    return int(round(number))


def clean_code(value):
    """Chuẩn hóa mã căn/mã lô dạng text."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\s+", " ", text).upper() if text else None


In [4]:
# Áp dụng parse numeric

df["price_vnd"]         = df["price"].apply(parse_price_vnd)
df["area_m2"]           = df["area"].apply(parse_area_m2)
df["land_area_m2"]      = df["Diện tích đất:"].apply(parse_area_m2)
df["price_per_m2_vnd"]  = df["Giá/m2:"].apply(parse_price_per_m2)
df["width_m"]           = df["Chiều ngang:"].apply(parse_meter)
df["length_m"]          = df["Chiều dài:"].apply(parse_meter)
df["bedrooms"]          = df["Số phòng ngủ:"].apply(parse_count)
df["toilets"]           = df["Số phòng vệ sinh:"].apply(parse_count)
df["usable_area_m2"]    = df["Diện tích sử dụng:"].apply(parse_area_m2)
df["apartment_area_m2"] = df["Diện tích:"].apply(parse_area_m2)
df["floors"]            = df["Tổng số tầng:"].apply(parse_count)
df["floor_number"]      = df["Tầng số:"].apply(parse_count)
df["unit_code_clean"]   = df["Mã căn / Mã căn hộ:"].apply(clean_code)
df["lot_code_clean"]    = df["Mã lô:"].apply(clean_code)

# Kiểm tra nhanh
df[["price", "price_vnd", "area", "area_m2", "Giá/m2:", "price_per_m2_vnd"]].head(5)

,price,price_vnd,area,area_m2,Giá/m2:,price_per_m2_vnd
0,"2,38 tỷ- 100 m2",2380000000,- 100 m2,100.0,"23,8 triệu/m2",23800000
1,18 tỷ- 79 m2,18000000000,- 79 m2,79.0,"227,85 triệu/m²",227850000
2,1 tỷ- 500 m2,1000000000,- 500 m2,500.0,2 triệu/m2,2000000
3,525 triệu- 60 m2,525000000,- 60 m2,60.0,"8,75 triệu/m²",8750000
4,440 triệu- 150 m2,440000000,- 150 m2,150.0,"2,93 triệu/m2",2930000


### 2a. Xử lý missing — Tổng số tầng

Drop các dòng không có `floors` vì đây là feature quan trọng và dataset đủ lớn để loại bỏ.

In [5]:
print("Missing floors:", df["floors"].isna().sum())
df = df.dropna(subset=["floors"]).copy()
print("Shape sau khi drop:", df.shape)

Missing floors: 12
Shape sau khi drop: (8992, 43)


### 2b. Scale giá về triệu VNĐ

Chia `price_vnd` cho 1,000,000 để dễ đọc và giảm chênh lệch scale với các feature khác.

In [6]:
df["Price (trieu VND)"] = df["price_vnd"] / 1_000_000
df[["price", "price_vnd", "Price (trieu VND)"]].head(5)

,price,price_vnd,Price (trieu VND)
12,"5,5 tỷ- 36.3 m2",5500000000,5500.0
13,"1,57 tỷ- 115 m2",1570000000,1570.0
14,580 triệu- 250 m2,580000000,580.0
15,1 tỷ- 3.031 m2,1000000000,1000.0
16,580 triệu- 440 m2,580000000,580.0


### 2c. Xử lý missing — Tầng số

`floor_number` không áp dụng cho mọi loại BĐS → giữ lại, fill `-1` = "không áp dụng",  
thêm cột flag `has_floor_number` để model phân biệt.

In [7]:
df["has_floor_number"] = df["floor_number"].notnull().astype(int)
df["floor_number"]     = df["floor_number"].fillna(-1)
print("Missing floor_number after fill:", df["floor_number"].isnull().sum())

Missing floor_number after fill: 0


### 2d. Log lỗi parse & flag inconsistency

In [8]:
def add_reason(current, reason):
    return reason if (pd.isna(current) or current == "") else str(current) + "; " + reason

df["parse_log"] = ""

log_rules = [
    ("price",              "price_vnd",         "parse_fail_price"),
    ("area",               "area_m2",            "parse_fail_area"),
    ("Diện tích đất:",     "land_area_m2",       "parse_fail_land_area"),
    ("Giá/m2:",            "price_per_m2_vnd",   "parse_fail_price_per_m2"),
    ("Chiều ngang:",       "width_m",            "parse_fail_width"),
    ("Chiều dài:",         "length_m",           "parse_fail_length"),
    ("Số phòng ngủ:",      "bedrooms",           "parse_fail_bedrooms"),
    ("Số phòng vệ sinh:",  "toilets",            "parse_fail_toilets"),
    ("Diện tích sử dụng:", "usable_area_m2",     "parse_fail_usable_area"),
    ("Diện tích:",         "apartment_area_m2",  "parse_fail_apartment_area"),
    ("Tổng số tầng:",      "floors",             "parse_fail_floors"),
    ("Tầng số:",           "floor_number",       "parse_fail_floor_number"),
]

for raw_col, clean_col, reason in log_rules:
    if raw_col not in df.columns:
        continue
    mask = df[raw_col].notna() & df[clean_col].isna()
    df.loc[mask, "parse_log"] = df.loc[mask, "parse_log"].apply(lambda x: add_reason(x, reason))

# Flag diện tích không nhất quán (width*length lệch >25% so với area)
calc_area = df["width_m"] * df["length_m"]
ref_area  = df["land_area_m2"].fillna(df["area_m2"])
ratio     = (calc_area - ref_area).abs() / ref_area
mask = calc_area.notna() & ref_area.notna() & (ref_area > 0) & (ratio > 0.25)
df.loc[mask, "parse_log"] = df.loc[mask, "parse_log"].apply(lambda x: add_reason(x, "inconsistent_area"))

# Flag đơn giá không nhất quán (price/area lệch >35% so với Giá/m2)
calc_ppm = df["price_vnd"] / df["area_m2"]
ppm_ratio = (calc_ppm - df["price_per_m2_vnd"]).abs() / df["price_per_m2_vnd"]
mask = calc_ppm.notna() & df["price_per_m2_vnd"].notna() & (df["price_per_m2_vnd"] > 0) & (ppm_ratio > 0.35)
df.loc[mask, "parse_log"] = df.loc[mask, "parse_log"].apply(lambda x: add_reason(x, "inconsistent_price_per_m2"))

df["parse_log"] = df["parse_log"].replace("", np.nan)
print("Rows có parse log:", df["parse_log"].notna().sum())

Rows có parse log: 2523


## 3. Parse Location

In [9]:
def normalize_text_location(text):
    """Normalize text cho location (giữ tiếng Việt có dấu)."""
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text.lower().strip())


def clean_location(text):
    if pd.isna(text):
        return None
    text = text.lower().strip()
    text = re.sub(r"\|\|\d+", "", text)
    return re.sub(r"\s+", " ", text)


def starts_with_any(text, keywords):
    return any(text.startswith(k + " ") or text == k for k in keywords)


def parse_location(text):
    if pd.isna(text):
        return pd.Series({"street": None, "ward": None, "district": None, "city": None})

    text  = normalize_text_location(text)
    parts = [p.strip() for p in text.split(",") if p.strip()]

    # bỏ số đầu (vd: "36")
    if len(parts) > 1 and re.fullmatch(r"\d+", parts[0]):
        parts = parts[1:]

    result = {"street": None, "ward": None, "district": None, "city": None}

    for part in parts:
        if starts_with_any(part, ["phường", "xã", "thị trấn", "thôn", "kênh"]):
            result["ward"] = part
        elif starts_with_any(part, ["quận", "huyện", "thị xã"]):
            result["district"] = part
        elif starts_with_any(part, ["đường"]):
            result["street"] = part
        elif starts_with_any(part, ["tỉnh", "tp"]):
            result["city"] = part
        elif part.startswith("thành phố"):
            if result["district"] is None:
                result["district"] = part
            else:
                result["city"] = part

    # Fallback city
    if result["city"] is None and len(parts) >= 3:
        result["city"] = parts[-1]

    used      = set(v for v in result.values() if v)
    remaining = [p for p in parts if p not in used]
    for key in ["street", "ward", "district"]:
        if result[key] is None and remaining:
            result[key] = remaining.pop(0)

    return pd.Series(result)


# Áp dụng
df["location_clean"] = df["location"].apply(clean_location)
df[["street", "ward", "district", "city"]] = df["location_clean"].apply(parse_location)
df = df.drop(columns=["location_clean"], errors="ignore")

df[["location", "street", "ward", "district", "city"]].head(10)

,location,street,ward,district,city
12,"Đường Lê Quang Định, Phường 11, Quận Bình Thạnh, Tp Hồ Chí Minh",đường lê quang định,phường 11,quận bình thạnh,tp hồ chí minh
13,"KHU DÂN CƯ MINH THẮNG - CÀ MAU, Phường 9, Thành phố Cà Mau, Cà Mau",khu dân cư minh thắng - cà mau,phường 9,thành phố cà mau,cà mau
14,"QL13, Thị trấn Lai Uyên, Huyện Bàu Bàng, Bình Dương",ql13,thị trấn lai uyên,huyện bàu bàng,bình dương
15,"Đường 765, Xã Sông Ray, Huyện Cẩm Mỹ, Đồng Nai",đường 765,xã sông ray,huyện cẩm mỹ,đồng nai
16,"Đường Tỉnh lộ 934, Xã Viên Bình, Huyện Trần Đề, Sóc Trăng",đường tỉnh lộ 934,xã viên bình,huyện trần đề,sóc trăng
17,"Đường Tô Ký, Phường Đông Hưng Thuận, Quận 12, Tp Hồ Chí Minh",đường tô ký,phường đông hưng thuận,quận 12,tp hồ chí minh
18,"Quốc Lộ 60, Xã Sơn Đông, Thành phố Bến Tre, Bến Tre",quốc lộ 60,xã sơn đông,thành phố bến tre,bến tre
19,"Đường Phan Huy Chú, Phường An Khánh, Quận Ninh Kiều, Cần Thơ",đường phan huy chú,phường an khánh,quận ninh kiều,cần thơ
20,"Đường Lý Thái Tổ, Phường Tân Lợi, Thành phố Buôn Ma Thuột, Đắk Lắk",đường lý thái tổ,phường tân lợi,thành phố buôn ma thuột,đắk lắk
21,"Vĩnh Quỳnh, Xã Ngọc Hồi, Huyện Thanh Trì, Hà Nội",vĩnh quỳnh,xã ngọc hồi,huyện thanh trì,hà nội


## 4. Clean Categorical Columns

In [10]:
# 4a. Fill Unknown cho các cột đơn giản 
simple_fill_cols = [
    "Loại hình nhà ở:",
    "Tình trạng nội thất:",
    "Tình trạng bất động sản:",
    "Hướng ban công:",
    "Đặc điểm căn hộ:",
    "Loại hình văn phòng:",
    "city", "district", "ward", "street",
]

for col in simple_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str).str.strip().replace("", "Unknown")

# 4b. Loại hình căn hộ — drop dòng "Nhà" không có loại căn hộ 
df = df[~(df["Loại hình căn hộ:"].isna() & df["title"].str.contains("Nhà", case=False, na=False))]
df["Loại hình căn hộ:"] = df["Loại hình căn hộ:"].fillna("Unknown").astype(str).str.strip().replace("", "Unknown")


In [11]:
#  4c. Tên phân khu / Lô / Block / Tháp 

def clean_text_block(x):
    if pd.isna(x):
        return x
    return re.sub(r"\s+", " ", x.upper().strip())


def classify_block_type(x):
    if pd.isna(x):
        return "other"
    if any(k in x for k in ["KHU", "DỰ ÁN", "VILLAGE", "CITY", "RESIDENCE", "RESIDENCES",
                              "PARADISE", "RIVERSIDE", "URBAN", "KDC"]):
        return "project"
    if re.fullmatch(r"[A-Z]\d{0,2}", x):
        return "block"
    if re.fullmatch(r"(LÔ|BLOCK)\s*[A-Z0-9]+", x):
        return "block"
    if re.fullmatch(r"[A-Z](\s+[A-Z0-9]+)+", x):
        return "block"
    return "other"


def normalize_block(x):
    if pd.isna(x):
        return x
    parts = sorted(set(re.split(r"[,\s]+", x)))
    return ",".join(parts)


raw_col = "Tên phân khu/Lô/Block/Tháp:"
df["_blk_clean"]  = df[raw_col].apply(clean_text_block)
df["_blk_type"]   = df["_blk_clean"].apply(classify_block_type)

df["Phân khu/Lô/Block/Tháp"] = None
df.loc[df["_blk_type"] == "project", "Phân khu/Lô/Block/Tháp"] = df["_blk_clean"]
df.loc[df["_blk_type"] == "block",   "Phân khu/Lô/Block/Tháp"] = df["_blk_clean"].apply(normalize_block)
df.loc[df["_blk_type"] == "other",   "Phân khu/Lô/Block/Tháp"] = "Không thuộc project/block"
df["Phân khu/Lô/Block/Tháp"] = df["Phân khu/Lô/Block/Tháp"].fillna("Không có thông tin")

df = df.drop(columns=["_blk_clean", "_blk_type"], errors="ignore")


## 5. Post-processing & Export

In [12]:
# 5a. Ghi cột clean vào cột gốc tương ứng
clean_to_original = {
    "price_vnd":          "price",
    "area_m2":            "area",
    "land_area_m2":       "Diện tích đất:",
    "price_per_m2_vnd":   "Giá/m2:",
    "width_m":            "Chiều ngang:",
    "length_m":           "Chiều dài:",
    "bedrooms":           "Số phòng ngủ:",
    "toilets":            "Số phòng vệ sinh:",
    "floors":             "Tổng số tầng:",
    "usable_area_m2":     "Diện tích sử dụng:",
    "apartment_area_m2":  "Diện tích:",
    "floor_number":       "Tầng số:",
    "unit_code_clean":    "Mã căn / Mã căn hộ:",
    "lot_code_clean":     "Mã lô:",
}
for clean_col, original_col in clean_to_original.items():
    if clean_col in df.columns and original_col in df.columns:
        df[original_col] = df[clean_col]

# ── 5b. Chọn cột final ───────────────────────────────────────────────────
raw_cols    = pd.read_csv(INPUT_PATH, nrows=0).columns.tolist()
extra_cols  = ["street", "ward", "district", "city", "Phân khu/Lô/Block/Tháp",
               "has_floor_number", "Price (trieu VND)"]
final_cols  = raw_cols + [c for c in extra_cols if c in df.columns]
df_final    = df[[c for c in final_cols if c in df.columns]].copy()

# ── 5c. Rename & drop cols ───────────────────────────────────────────────
df_final = df_final.rename(columns={"price": "price_vnd"})
df_final["price_vnd"]         = df_final["price_vnd"].astype("Int64")
df_final["Price (trieu VND)"] = df_final["Price (trieu VND)"].astype(float)

drop_cols = [
    "price_vnd",                    # đã có Price (trieu VND) làm target
    "Giá/m2:",                      # có thể tính lại từ price/area
    "description",                  # text dài, không dùng cho model
    "location",                     # đã có street/ward/district/city
    "Mã căn / Mã căn hộ:",         # identifier
    "Mã lô:",                       # identifier
    "Tên phân khu/Lô/Block/Tháp:", # đã có Phân khu/Lô/Block/Tháp
]
df_final = df_final.drop(columns=[c for c in drop_cols if c in df_final.columns])
print("Sau drop cols:", df_final.shape)

# ── 5d. Drop duplicates ──────────────────────────────────────────────────
before = len(df_final)
df_final = df_final.drop_duplicates(subset=["title", "Price (trieu VND)", "area"], keep="first")
print(f"Duplicates dropped: {before - len(df_final)}")

# ── 5e. Drop outlier area (giữ NaN) ─────────────────────────────────────
p1  = df_final["area"].quantile(0.01)
p99 = df_final["area"].quantile(0.99)
mask_ok  = df_final["area"].isna() | df_final["area"].between(p1, p99)
before   = len(df_final)
df_final = df_final[mask_ok]
print(f"Area bounds: {p1:,.1f} → {p99:,.1f} m² | Outliers dropped: {before - len(df_final)}")


Sau drop cols: (8992, 29)
Duplicates dropped: 1890
Area bounds: 20.0 → 7,448.9 m² | Outliers dropped: 133


In [13]:
# ── 5f. Summary & Export ─────────────────────────────────────────────────
print("=== SUMMARY ===")
print(f"Final shape:       {df_final.shape}")
print(f"Price (trieu VND): {df_final['Price (trieu VND)'].min():,.1f} → {df_final['Price (trieu VND)'].max():,.1f} triệu VNĐ")
print(f"area:              {df_final['area'].min():.1f} → {df_final['area'].max():.1f} m²  |  NaN: {df_final['area'].isna().sum()}")

missing = df_final.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    print(f"\nMissing:\n{missing}")

print(f"\nColumns: {df_final.columns.tolist()}")

df_final.to_csv(OUTPUT_FINAL, index=False, encoding="utf-8-sig")
print(f"\n Exported: {OUTPUT_FINAL}")


=== SUMMARY ===
Final shape:       (6969, 29)
Price (trieu VND): 0.4 → 860,000.0 triệu VNĐ
area:              20.0 → 7408.0 m²  |  NaN: 362

Missing:
area    362
dtype: int64

Columns: ['title', 'area', 'Diện tích đất:', 'Hướng cửa chính:', 'Giấy tờ pháp lý:', 'Đặc điểm nhà/đất:', 'Loại hình đất:', 'Chiều ngang:', 'Chiều dài:', 'Số phòng ngủ:', 'Số phòng vệ sinh:', 'Loại hình nhà ở:', 'Tình trạng nội thất:', 'Diện tích sử dụng:', 'Tình trạng bất động sản:', 'Diện tích:', 'Loại hình căn hộ:', 'Tổng số tầng:', 'Tầng số:', 'Hướng ban công:', 'Đặc điểm căn hộ:', 'Loại hình văn phòng:', 'street', 'ward', 'district', 'city', 'Phân khu/Lô/Block/Tháp', 'has_floor_number', 'Price (trieu VND)']

 Exported: ..\data\chotot_final.csv
